# WildJailbreak EDA

Exploratory Data Analysis для датасета WildJailbreak.

Секции:
1. Загрузка и базовая статистика датасета
2. Распределение классов (train и eval)
3. Распределение длин промптов по классам
4. Примеры из каждого data_type
5. Сравнение vanilla vs adversarial промптов
6. Few-shot сплиты: таблица размеров

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Загрузка и базовая статистика датасета

In [ ]:
# Загрузка raw данных
data_dir = Path("../data")

def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

# TODO: загрузить train и eval
# train_df = load_jsonl(data_dir / "raw" / "wildjailbreak_train.jsonl")
# eval_df = load_jsonl(data_dir / "raw" / "wildjailbreak_eval.jsonl")

In [ ]:
# Базовая статистика
# print(f"Train: {len(train_df)} examples")
# print(f"Eval:  {len(eval_df)} examples")
# print(f"\nКолонки: {list(train_df.columns)}")

## 2. Распределение классов (train и eval)

Отдельно по data_type и по binary_label.

In [ ]:
# Добавление binary_label
def get_binary_label(data_type):
    if data_type in {"vanilla_harmful", "adversarial_harmful"}:
        return "jailbreak"
    return "safe"

# train_df["binary_label"] = train_df["data_type"].apply(get_binary_label)
# eval_df["binary_label"] = eval_df["data_type"].apply(get_binary_label)

In [ ]:
# Распределение по data_type
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# 
# train_df["data_type"].value_counts().plot(kind="bar", ax=axes[0], title="Train: data_type")
# eval_df["data_type"].value_counts().plot(kind="bar", ax=axes[1], title="Eval: data_type")
# 
# plt.tight_layout()
# plt.show()

In [ ]:
# Распределение по binary_label
# fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# 
# train_df["binary_label"].value_counts().plot(kind="bar", ax=axes[0], title="Train: binary_label")
# eval_df["binary_label"].value_counts().plot(kind="bar", ax=axes[1], title="Eval: binary_label")
# 
# plt.tight_layout()
# plt.show()

## 3. Распределение длин промптов по классам

In [ ]:
# Функция для получения prompt
def get_prompt(row):
    adversarial = row.get("adversarial", "")
    if adversarial and str(adversarial).strip():
        return adversarial
    return row["vanilla"]

# train_df["prompt"] = train_df.apply(get_prompt, axis=1)
# train_df["prompt_len"] = train_df["prompt"].str.len()
# 
# eval_df["prompt"] = eval_df.apply(get_prompt, axis=1)
# eval_df["prompt_len"] = eval_df["prompt"].str.len()

In [ ]:
# Гистограмма длин по binary_label
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# 
# for label in ["safe", "jailbreak"]:
#     subset = eval_df[eval_df["binary_label"] == label]
#     axes[0].hist(subset["prompt_len"], bins=50, alpha=0.6, label=label)
# axes[0].set_title("Eval: Prompt Length Distribution")
# axes[0].set_xlabel("Length (chars)")
# axes[0].legend()
# 
# # Box plot
# eval_df.boxplot(column="prompt_len", by="binary_label", ax=axes[1])
# axes[1].set_title("Eval: Prompt Length by Class")
# 
# plt.tight_layout()
# plt.show()

## 4. Примеры из каждого из 4 data_type

In [ ]:
# Примеры из каждого data_type
# for dtype in ["vanilla_benign", "vanilla_harmful", "adversarial_benign", "adversarial_harmful"]:
#     print(f"\n=== {dtype} ===")
#     subset = eval_df[eval_df["data_type"] == dtype]
#     if len(subset) > 0:
#         example = subset.iloc[0]
#         print(f"Vanilla: {example['vanilla'][:200]}...")
#         if example.get('adversarial'):
#             print(f"Adversarial: {str(example['adversarial'])[:200]}...")
#     print()

## 5. Сравнение vanilla vs adversarial промптов для одного harmful примера

In [ ]:
# Найти adversarial_harmful пример с непустым adversarial
# adversarial_examples = eval_df[
#     (eval_df["data_type"] == "adversarial_harmful") & 
#     (eval_df["adversarial"].notna()) &
#     (eval_df["adversarial"].str.len() > 0)
# ]
# 
# if len(adversarial_examples) > 0:
#     example = adversarial_examples.iloc[0]
#     print("=== Vanilla (original harmful prompt) ===")
#     print(example["vanilla"])
#     print("\n=== Adversarial (jailbreak version) ===")
#     print(example["adversarial"])
#     print("\n=== Tactics ===")
#     print(example.get("tactics", "N/A"))

## 6. Few-shot сплиты: таблица n_shot x seed с размерами

In [ ]:
# Загрузка few-shot сплитов и подсчёт размеров
# processed_dir = data_dir / "processed"
# 
# n_shots = [10, 20, 50]
# seeds = [42, 123, 456]
# 
# results = []
# for n in n_shots:
#     for seed in seeds:
#         path = processed_dir / f"train_shot{n}_seed{seed}.json"
#         if path.exists():
#             with open(path, "r") as f:
#                 data = json.load(f)
#             n_safe = len(data["intents"][0]["utterances"])
#             n_jailbreak = len(data["oos_utterances"])
#             results.append({
#                 "n_shots": n,
#                 "seed": seed,
#                 "safe": n_safe,
#                 "jailbreak": n_jailbreak,
#                 "total": n_safe + n_jailbreak
#             })
# 
# pd.DataFrame(results)

In [ ]:
# Pivot table для наглядности
# df_results = pd.DataFrame(results)
# pivot = df_results.pivot(index="n_shots", columns="seed", values="total")
# pivot